# Connect AI — 장기 기억 학습

이 공개 노트북에는 지식 원문이나 토큰이 포함되어 있지 않습니다.

실행 시:
1. Hugging Face에 로그인합니다.
2. 비공개 데이터셋 `willseungmin/connect-ai-brain`을 읽습니다.
3. Qwen2.5 0.5B를 LoRA 방식으로 학습합니다.
4. 결과를 `willseungmin/my-brain-v1`에 safetensors와 GGUF로 업로드합니다.

**런타임 → 런타임 유형 변경 → T4 GPU**, 그다음 **런타임 → 모두 실행**을 누르세요.


In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo datasets trl huggingface_hub


## Hugging Face 로그인

Write 권한 토큰으로 로그인하세요. 토큰은 노트북이나 GitHub에 저장되지 않습니다.


In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()
print("로그인 계정:", HfApi().whoami()["name"])


In [ ]:
DATASET_REPO = "willseungmin/connect-ai-brain"
DATASET_FILE = "connect-ai-brain.jsonl"
OUTPUT_REPO = "willseungmin/my-brain-v1"
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct"
MAX_STEPS = 40


In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    dtype=None,
    max_seq_length=1024,
    load_in_4bit=True,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    finetune_vision_layers=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)
print("베이스 모델 로딩 완료")


In [ ]:
from datasets import load_dataset

ds = load_dataset(
    DATASET_REPO,
    data_files=DATASET_FILE,
    split="train",
    token=True,
)

def format_batch(batch):
    texts = [
        tokenizer.apply_chat_template(
            conversation,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        for conversation in batch["conversations"]
    ]
    return {"text": texts}

ds = ds.map(format_batch, batched=True)
print("학습 데이터 수:", len(ds))
print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=MAX_STEPS,
        learning_rate=3e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=1e-3,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)

sample = ds[0]["text"]
if "<|im_start|>" in sample:
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

stats = trainer.train()
print("학습 완료, loss:", round(stats.training_loss, 4))


In [ ]:
from unsloth import FastModel
FastModel.for_inference(model)

def chat(prompt, max_tokens=220):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    print(answer)

chat("내 사업과 지식에 대해 아는 내용을 요약해줘.")


In [ ]:
import gc, torch
from huggingface_hub import HfApi

try:
    del trainer
except Exception:
    pass

gc.collect()
torch.cuda.empty_cache()

owner = HfApi().whoami()["name"]
if not OUTPUT_REPO.startswith(owner + "/"):
    raise ValueError(f"OUTPUT_REPO 소유자가 로그인 계정({owner})과 다릅니다: {OUTPUT_REPO}")

try:
    model.push_to_hub_merged(
        OUTPUT_REPO,
        tokenizer,
        save_method="merged_16bit",
        token=True,
    )
    print("safetensors 업로드 완료")
except Exception as exc:
    print("병합 업로드 실패, LoRA로 폴백:", exc)
    model.push_to_hub(OUTPUT_REPO, token=True)
    tokenizer.push_to_hub(OUTPUT_REPO, token=True)

model.push_to_hub_gguf(
    OUTPUT_REPO,
    tokenizer,
    quantization_method="q4_k_m",
    token=True,
)
print("완료:", f"https://huggingface.co/{OUTPUT_REPO}")
